# ⚡ Pokémon Legendary KNN Classifier
## K-Nearest Neighbor Mini-System Prototype
**Course:** Data Mining and Business Intelligence

---
**System Goal:** Classify a Pokémon as Legendary or Non-Legendary using its base stats and the KNN algorithm.

**Target Variable:** `Class` — Legendary / Non-Legendary

**Features:** HP, Attack, Defense, Sp_Atk, Sp_Def, Speed

## Step 0 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, classification_report
)
import warnings
warnings.filterwarnings('ignore')
print('✅ Libraries imported successfully')

## Step 1 — Load the Dataset

In [ ]:
# ── Change this path to point to your CSV file ──
CSV_PATH = '../data/knn_dataset.csv'

df = pd.read_csv(CSV_PATH)
print(f'Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'Columns: {list(df.columns)}')
df.head(10)

## Step 2 — Dataset Exploration

In [ ]:
# Basic info
print('=== Data Types ===' )
print(df.dtypes)
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Class Distribution ===')
print(df['Class'].value_counts())

In [ ]:
# Descriptive statistics
df.describe().round(2)

In [ ]:
# Class distribution chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
dist = df['Class'].value_counts()
colors = ['#FFCB05', '#2A75BB']
bars = axes[0].bar(dist.index, dist.values, color=colors, edgecolor='white', linewidth=2)
axes[0].bar_label(bars, fmt='%d', padding=3, fontweight='bold')
axes[0].set_title('Class Distribution', fontweight='bold')
axes[0].set_ylabel('Count')

# Pie chart
axes[1].pie(dist.values, labels=dist.index, colors=colors,
            autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Class Proportion', fontweight='bold')

plt.suptitle('Pokémon Class Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 3 — Data Preparation

In [ ]:
# Define feature columns and target
FEATURE_COLS = ['HP', 'Attack', 'Defense', 'Sp_Atk', 'Sp_Def', 'Speed']
TARGET_COL   = 'Class'

# Drop rows with missing values in selected columns
df_clean = df[FEATURE_COLS + [TARGET_COL]].dropna()
print(f'Records after cleaning: {len(df_clean)} (removed {len(df) - len(df_clean)} rows with NaN)')

X = df_clean[FEATURE_COLS].values
y = df_clean[TARGET_COL].values

print(f'X shape: {X.shape}')
print(f'y classes: {np.unique(y)}')

## Step 4 — Train-Test Split

In [ ]:
# Split: 80% training, 20% testing
# Fixed random_state=42 ensures reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.80, random_state=42, stratify=y
)

print(f'Training set : {len(X_train)} records ({len(X_train)/len(X):.0%})')
print(f'Test set     : {len(X_test)} records ({len(X_test)/len(X):.0%})')

## Step 5 — Feature Scaling (Min-Max Normalization)

**Why scale?** KNN uses distances. If HP ranges from 1–255 and Speed from 1–200,
HP would dominate the distance computation. Normalization brings all features to [0, 1].

> Formula: `x_scaled = (x - min) / (max - min)`

⚠️ **Important:** Fit the scaler ONLY on training data. Apply it to both train and test.

In [ ]:
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit + transform training
X_test_scaled  = scaler.transform(X_test)         # transform ONLY (no fit)

print('Before scaling (X_train first row):', X_train[0])
print('After scaling  (X_train first row):', X_train_scaled[0].round(4))

## Step 6 — Train the KNN Classifier

In [ ]:
K = 5  # ← Change this value to experiment (try 1, 3, 5, 7, 9)

# Validate K
assert K > 0, 'K must be positive'
assert K <= len(X_train_scaled), f'K ({K}) cannot exceed training size ({len(X_train_scaled)})'

model = KNeighborsClassifier(n_neighbors=K, metric='euclidean')
model.fit(X_train_scaled, y_train)

print(f'✅ KNN model trained with K={K}')

## Step 7 — Evaluate on Test Set

In [ ]:
y_pred = model.predict(X_test_scaled)

# Metrics
acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, pos_label='Legendary', zero_division=0)
rec  = recall_score(y_test, y_pred, pos_label='Legendary', zero_division=0)
f1   = f1_score(y_test, y_pred, pos_label='Legendary', zero_division=0)

print(f'K = {K}')
print(f'Accuracy  : {acc:.4f} ({acc:.1%})')
print(f'Precision : {prec:.4f}')
print(f'Recall    : {rec:.4f}')
print(f'F1-Score  : {f1:.4f}')

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
labels = sorted(np.unique(y_test))

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrBr',
            xticklabels=labels, yticklabels=labels,
            linewidths=2, linecolor='white', ax=ax,
            annot_kws={'size': 14, 'weight': 'bold'})
ax.set_xlabel('Predicted Label', fontweight='bold')
ax.set_ylabel('Actual Label', fontweight='bold')
ax.set_title(f'Confusion Matrix (K={K})', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Full classification report
print('\n=== Classification Report ===')
print(classification_report(y_test, y_pred))

## Step 8 — Classify a New Pokémon Record

This demonstrates how the system handles a completely new, unseen record.

In [ ]:
# ── Enter stats for the new Pokémon here ──
new_pokemon = {
    'Name'   : 'MysterMon',
    'HP'     : 106,
    'Attack' : 110,
    'Defense': 90,
    'Sp_Atk' : 154,
    'Sp_Def' : 90,
    'Speed'  : 130,
}

# Prepare input vector
query_raw    = np.array([[new_pokemon[f] for f in FEATURE_COLS]])
query_scaled = scaler.transform(query_raw)

# Predict
predicted = model.predict(query_scaled)[0]
print(f"Pokémon : {new_pokemon['Name']}")
print(f"Stats   : {query_raw[0]}")
print(f"\n>>> Predicted Class: {predicted} <<<")

In [ ]:
# Show nearest neighbors manually
def euclidean_distance(a, b):
    return np.sqrt(np.sum((a - b) ** 2))

query_vec = query_scaled[0]
distances = [(euclidean_distance(query_vec, X_train_scaled[i]), y_train[i], i)
             for i in range(len(X_train_scaled))]
distances.sort(key=lambda x: x[0])

print(f'\n=== Top {K} Nearest Neighbors ===')
print(f'{"Rank":<6} {"Distance":<12} {"Class":<20}')
print('-' * 40)
legend_votes = 0
for rank, (dist, cls, idx) in enumerate(distances[:K], 1):
    if cls == 'Legendary':
        legend_votes += 1
    print(f'{rank:<6} {dist:<12.5f} {cls:<20}')

non_legend_votes = K - legend_votes
print(f'\nVotes → Legendary: {legend_votes} | Non-Legendary: {non_legend_votes}')
print(f'Majority vote → {"Legendary" if legend_votes > non_legend_votes else "Non-Legendary"}')

## Step 9 — Bonus: K Value Comparison

In [ ]:
k_values = [1, 3, 5, 7, 9, 11]
results = []

for k in k_values:
    clf = KNeighborsClassifier(n_neighbors=k, metric='euclidean')
    clf.fit(X_train_scaled, y_train)
    preds = clf.predict(X_test_scaled)
    results.append({
        'K': k,
        'Accuracy': accuracy_score(y_test, preds),
        'F1-Score': f1_score(y_test, preds, pos_label='Legendary', zero_division=0)
    })

results_df = pd.DataFrame(results)
print(results_df.round(4))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(results_df['K'], results_df['Accuracy'], marker='o', label='Accuracy',
        color='#2A75BB', linewidth=2)
ax.plot(results_df['K'], results_df['F1-Score'], marker='s', label='F1-Score',
        color='#FFCB05', linewidth=2)
ax.set_xlabel('K Value', fontweight='bold')
ax.set_ylabel('Score', fontweight='bold')
ax.set_title('Accuracy and F1-Score by K Value', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()